In [41]:
import pandas as pd
import re
import string
import unicodedata
import emoji

pd.set_option("display.max_colwidth", None)

In [43]:
df = pd.read_csv("EDA_dataset.csv")
df.head()

,reviewId,content,score,at,bank,year
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,"ribet banget ni apk sumpah dikir' verif dikit' verif udhmah nyedot pulsa mulu, tolong benerin lah, jangan ribet gini mau pake aja ribet cuh",1,2025-12-31 23:57:59,BCAMOBILE_REVIEWS,2025
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,kenapa QRis ga bisa di pakai ya?? daritadi loading mulu. transfer uang juga gabisa,2,2025-12-31 23:22:32,BCAMOBILE_REVIEWS,2025
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,"aplikasi nya sampah, kenapa tiba-tiba keluar terus registrasi ulang lagi, harus pake SMS pulsa pula, sangat tidak membantu ,delete",1,2025-12-31 23:06:58,BCAMOBILE_REVIEWS,2025
3,32d28ac6-c538-4749-969f-8af060915d96,bagus,3,2025-12-31 21:47:45,BCAMOBILE_REVIEWS,2025
4,f99259b7-0d45-4422-b667-6c4fd521638b,tidak ada solusi ketika ada kendala di persulit di lempar kesana kesini sangat mengecewakan harusnya respon dan solusi yang di berikan,1,2025-12-31 21:18:21,BCAMOBILE_REVIEWS,2025


empty review removal

In [44]:
print(f"Before : {len(df):,}")

df = df[
    df["content"]
    .fillna("")
    .str.strip()
    .ne("")
].copy()

print(f"After  : {len(df):,}")

Before : 60,170
After  : 60,170


Noise Review Detection (Inspection Only)

In [45]:
noise_pattern = r"^[A-Za-z\s.,!?;:'\"()\-/\\]+$"

potential_noise = df[
    df["content"]
    .str.fullmatch(noise_pattern, na=False)
]

potential_noise.sample(20, random_state=42)

,reviewId,content,score,at,bank,year
20711,07fe9f35-1e81-4736-a7eb-ecdd6e9f5314,"Kenapa setiap saya mau masuk ngak bisa, udah saya coba beberapa kali tetap ngak bisa",2,2025-06-07 09:19:45,BRIMO_REVIEWS,2025
49296,5d7d5e93-b9f6-4f33-b224-f3f44ab1639e,lemooottttttt,1,2025-12-01 13:54:54,WONDR_BNI_REVIEWS,2025
15578,eb090978-2373-46e4-9409-50f97f9fce38,"Gak bisa login, Woi.",1,2025-08-27 20:38:38,BRIMO_REVIEWS,2025
51282,2a364f9d-6aa7-4be4-ac53-06ff23f7de14,"Transfer tdk bisa digunakan hari ini, dr pagi hingga sore.",1,2025-10-19 14:21:59,WONDR_BNI_REVIEWS,2025
4305,7258c684-ab1e-42df-8e3e-797871e3b123,GAK AMAN,1,2025-08-14 17:52:19,BCAMOBILE_REVIEWS,2025
17692,b654dc49-876e-4bf2-abc6-c0c5c8d328de,Transaksi Qris sering gagal pengajuan proses lama dan belum pasti saldo yang sudah terpotong kembali . Terimakasih,1,2025-08-01 22:00:05,BRIMO_REVIEWS,2025
28937,a1191680-b9f0-4c0c-b1ca-2c9548b04abb,mengapa bri berbeda dengan bca sedangkan bca saldo. bisa di habiskan,2,2025-01-13 00:07:33,BRIMO_REVIEWS,2025
9309,ef7599d1-f4c5-41ba-ae4b-d60dab1f39cc,"terkadang untuk masuk apk ini susah tiba tiba ponsel gue gak ada login layar putih semua,padahal jaringan internetnya ok",1,2025-12-26 11:07:47,BRIMO_REVIEWS,2025
37023,242b4814-5bbf-4a12-9918-8bcedd555cdf,"knp akhir""nie selalu di update trus aplikasi nya bikin pusing",2,2025-07-12 05:45:21,LIVIN_MANDIRI_REVIEWS,2025
45522,32f3e3a0-ca25-4b9a-824d-5e3284f2f53c,Kenapa saya tidak bisa login ya padahal no kartu sudah Benar Dan tanggal lahir,1,2025-01-19 19:55:49,LIVIN_MANDIRI_REVIEWS,2025


that weird reviews in eda phase isn't here, weird.

case folding

In [46]:
df["case_folding"] = (
    df["content"]
    .astype(str)
    .str.lower()
)

df[
    ["content", "case_folding"]
].head(10)

,content,case_folding
0,"ribet banget ni apk sumpah dikir' verif dikit' verif udhmah nyedot pulsa mulu, tolong benerin lah, jangan ribet gini mau pake aja ribet cuh","ribet banget ni apk sumpah dikir' verif dikit' verif udhmah nyedot pulsa mulu, tolong benerin lah, jangan ribet gini mau pake aja ribet cuh"
1,kenapa QRis ga bisa di pakai ya?? daritadi loading mulu. transfer uang juga gabisa,kenapa qris ga bisa di pakai ya?? daritadi loading mulu. transfer uang juga gabisa
2,"aplikasi nya sampah, kenapa tiba-tiba keluar terus registrasi ulang lagi, harus pake SMS pulsa pula, sangat tidak membantu ,delete","aplikasi nya sampah, kenapa tiba-tiba keluar terus registrasi ulang lagi, harus pake sms pulsa pula, sangat tidak membantu ,delete"
3,bagus,bagus
4,tidak ada solusi ketika ada kendala di persulit di lempar kesana kesini sangat mengecewakan harusnya respon dan solusi yang di berikan,tidak ada solusi ketika ada kendala di persulit di lempar kesana kesini sangat mengecewakan harusnya respon dan solusi yang di berikan
5,masa gue setiap isi Flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue!,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue!
6,Mau Aktifasi di Handphone baru semakin ribet dan sulit tidak seperti yang lalu,mau aktifasi di handphone baru semakin ribet dan sulit tidak seperti yang lalu
7,"sinyal penuh, internet lancar tapi masih aj kode merah... aneh bener","sinyal penuh, internet lancar tapi masih aj kode merah... aneh bener"
8,sinyal bagus tetap biru terus..membuat lama pembayaran,sinyal bagus tetap biru terus..membuat lama pembayaran
9,saya mau login menggunakan pulsa kode selalu tidak keluar,saya mau login menggunakan pulsa kode selalu tidak keluar


checking the data first before continuing 

In [47]:
df[
    df["case_folding"].str.contains(
        r"http|www\.",
        regex=True,
        na=False
    )
][["content"]]

,content


In [48]:
df[
    df["case_folding"].str.contains(
        r"<.*?>",
        regex=True,
        na=False
    )
][["content"]]

,content


In [49]:
df[
    df["case_folding"].str.contains(
        "@",
        regex=False,
        na=False
    )
][["content"]]

,content
3006,Masa kalah sama bank J@g0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.BCA Transaksi Qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan Saldo belum balik lagi . telpon Cs jawabannya slalu sama semua
10844,"pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian @BRI"
12010,"saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact BRI, via Wa, @mail hasilnya nihil. kacau ni brimo"
12460,f5.YG SUDAH BAYAR 50K 1.BPK KADIM 200 2.HUMAM 150 3.ADI GENDUT 100 4.HERI 50 5.PANGAT 50 6.AJIS 50 7.ILHAM 50 8 8.FAISAL 50 9.AGENG @ Ono ceritane cah ndugal mlayu Seko pacobaning Urip.List baju 1.wono 19(L) 120 2 Araujoo 27 (M) 120 3 Ageng 12 M 4 piszz 23(S DEWASA) 5 Y. R 14 (m) 6 R. Y 4 (L) 7 ILHAM 22 50 9.AGENG 50 10.DEWAN 50 11.NANOK 50 12.WAWAN 50 136556n6
12960,saya melakukan transfer di aplikasi brimo dengan menggunakan @nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata RANDOM dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii....😭😭😭😭😭😭 9 jt
15361,Kenapa login gagal trs ya @brimo setelah download lagi di beda negara. Boleh tolong saya kak
16564,apk mobil banking paling parah! transfer sesama bri pake @ alias2 Gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik
16629,"dapat notif : We detected something blocking the application screen. To make sure your data is safe, please close all popups and other overlays. If nothing is obstructing the application screen, try to restart the application and perform a malware scan. sudah sampai install ulang, reeboot, bahkan juga ke CS BRI langsung tetap tidak ada solusi, tutorial youtube, dll dkk, tetep tidak bisa buat transaksi dan muncul notif itu😭😭 #BRImo @BRImo"
17961,"saya mau isi saldo shoopay kok susah sekali ya, kalau mau isi saldo 2 juta aja ribet amat, hari ini trfer 1 juta pkek @wallet bisa, ntar kelang brpa jam gk bisa lagi. trz aku tfer 5rtus ribu bis, stlh itu gk bisa lagi, trz kirim lagi 3ratus bisa, trs sisanya mau trfer 2 ratus gk bisa lagi, jadi aku tf 100rbu bisa, mau tf lgi gk bisa lagi. adu pusing ya susah bner mau trfer ke shoopay aja ribet gitu. sebelumnya gk ada masalah mau tf brpa aja ke shoopay. sekarang ribet pusing gk tau salahnya dimna"
19907,"luar bi@sa semenjak menggunakan BRimo mudah melakukan transaksi, mantap"


unicoe normalization  
normalizing weird fonts

In [50]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return text
    return unicodedata.normalize("NFKC", text)

df["unicode_normalized"] = df["case_folding"].apply(normalize_unicode)

unicode_changed = df[
    df["case_folding"] != df["unicode_normalized"]
]

print(f"Total changed rows: {len(unicode_changed):,}")

unicode_changed[
    [
        "case_folding",
        "unicode_normalized"
    ]
].head(10)

Total changed rows: 1,325


,case_folding,unicode_normalized
20,"indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang² dan sering","indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang2 dan sering"
21,"kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang² trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya","kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang2 trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya"
28,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar² pake qris juga jadi males,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar2 pake qris juga jadi males
104,"saldo saya tiba² ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan, apk aneh","saldo saya tiba2 ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan, apk aneh"
117,"kenapa lemot sekali mau buka apk nya..pdhal jaringan bagus,tpi jaringan terputus terus kira² knpa ya apa lagi eror","kenapa lemot sekali mau buka apk nya..pdhal jaringan bagus,tpi jaringan terputus terus kira2 knpa ya apa lagi eror"
132,tau² oneklik dinonaktifkan aja ngga jelas apa alasannya.masa iya hanya sebulan ngga transaksi langsung diblokir..ngeselin,tau2 oneklik dinonaktifkan aja ngga jelas apa alasannya.masa iya hanya sebulan ngga transaksi langsung diblokir..ngeselin
305,"setelah update sering muncul indikator merah, sebelumnya lancar² saja dan qris loading sangat lama, bahkan sama sekali ga terbuka. mohon diperbaiki.","setelah update sering muncul indikator merah, sebelumnya lancar2 saja dan qris loading sangat lama, bahkan sama sekali ga terbuka. mohon diperbaiki."
341,indikator warna yg mengganggu sinyal ga cocok dikit gabisa dibuat apa² apknya,indikator warna yg mengganggu sinyal ga cocok dikit gabisa dibuat apa2 apknya
407,"kebiasaan kali sering gangguan pas malam minggu,udah gitu lampu indikator merahnya lama lgi anj,sebelum update aja dah mntp,gk usah di-update klw malah nambah masalah,mbankking dipakek cuma buat byr,ngirim scan brcode doang..gk usah diperberat atau nambah² inj itu jnck","kebiasaan kali sering gangguan pas malam minggu,udah gitu lampu indikator merahnya lama lgi anj,sebelum update aja dah mntp,gk usah di-update klw malah nambah masalah,mbankking dipakek cuma buat byr,ngirim scan brcode doang..gk usah diperberat atau nambah2 inj itu jnck"
432,"semenjak update versi terbaru jadi sangat² berat, mau cek saldo aja lama dan melakukan transaksi juga lama.. mohon segera diperbaiki","semenjak update versi terbaru jadi sangat2 berat, mau cek saldo aja lama dan melakukan transaksi juga lama.. mohon segera diperbaiki"


remove emoji

In [51]:
df["remove_emoji"] = (
    df["unicode_normalized"]
    .apply(lambda text: emoji.replace_emoji(text, replace=""))
)

In [52]:
emoji_changed = df[
    df["unicode_normalized"] != df["remove_emoji"]
][["unicode_normalized", "remove_emoji"]]

print(f"Rows affected: {len(emoji_changed)}")

emoji_changed.head(10)

Rows affected: 3206


,unicode_normalized,remove_emoji
54,"yg terupdate tiap login kenapa mesti 2x siih? login pertama setelah input kode akses, pasti auto belum masuk dan harus ulangi ketik lagi barulah bisa masuk. selalu kaya gitu lohh di versi terbaru ini. padahal versi sebelumnya gak pernah gini. normal aja sekali login langsung masuk. mohon penjelasannya min 🙏","yg terupdate tiap login kenapa mesti 2x siih? login pertama setelah input kode akses, pasti auto belum masuk dan harus ulangi ketik lagi barulah bisa masuk. selalu kaya gitu lohh di versi terbaru ini. padahal versi sebelumnya gak pernah gini. normal aja sekali login langsung masuk. mohon penjelasannya min"
75,habia update malah qris nya lola 🤦,habia update malah qris nya lola
85,mohon bertanya min saya pindah kota ke jawa ko mobile bca sya gk bisa di buka suruh masuk nomor kartu atm tidak ada konfirmasi lewat nomor hp min susah mau isi saldo mlhn eror begini min🙇,mohon bertanya min saya pindah kota ke jawa ko mobile bca sya gk bisa di buka suruh masuk nomor kartu atm tidak ada konfirmasi lewat nomor hp min susah mau isi saldo mlhn eror begini min
87,"mau login ke bca mobile susah, sampai harus ber puluh x tetep aj gak bisa padahal pulsa banyak. gak ad sulusi solusi bantuan g7🤨","mau login ke bca mobile susah, sampai harus ber puluh x tetep aj gak bisa padahal pulsa banyak. gak ad sulusi solusi bantuan g7"
114,"registrasi susah. gagal trs, jelek banget bca mobile skrg 👎👎","registrasi susah. gagal trs, jelek banget bca mobile skrg"
138,"kenapa di hp saya tidak bisa di buka , alasan jaringan bermasalah padhal wifi saya bagus mohon di perbaiki jika sudah saya akan ganti ke ⭐5","kenapa di hp saya tidak bisa di buka , alasan jaringan bermasalah padhal wifi saya bagus mohon di perbaiki jika sudah saya akan ganti ke 5"
148,aku gak bisa masuk akun lama aku 😭😖🙏,aku gak bisa masuk akun lama aku
190,bca mobile terblokir tanpa sebab musabab ada kesalahan pin atau apalah. akhirnya ngrepotin nasabah harus antri panjang ke cs💩💩,bca mobile terblokir tanpa sebab musabab ada kesalahan pin atau apalah. akhirnya ngrepotin nasabah harus antri panjang ke cs
198,"keluar"" mlu dh verifikasi mlu sms😠ud verifikasi wajah masi gagal mlu pulsa habis buat verifikasi bca doang","keluar"" mlu dh verifikasi mlu smsud verifikasi wajah masi gagal mlu pulsa habis buat verifikasi bca doang"
213,"mohon maaf ya bintang satu dulu, notifikasi transfer masuk tidak muncul☺","mohon maaf ya bintang satu dulu, notifikasi transfer masuk tidak muncul"


remove "@"

In [53]:
df["remove_at"] = (
    df["remove_emoji"]
    .str.replace("@", "", regex=False)
)

In [54]:
at_changed = df[
    df["remove_emoji"] != df["remove_at"]
][["remove_emoji", "remove_at"]]

print(f"Rows affected: {len(at_changed):,}")

at_changed.head(10)

Rows affected: 31


,remove_emoji,remove_at
3006,masa kalah sama bank j@g0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.bca transaksi qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan saldo belum balik lagi . telpon cs jawabannya slalu sama semua,masa kalah sama bank jg0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.bca transaksi qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan saldo belum balik lagi . telpon cs jawabannya slalu sama semua
10844,"pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian @bri","pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian bri"
12010,"saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact bri, via wa, @mail hasilnya nihil. kacau ni brimo","saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact bri, via wa, mail hasilnya nihil. kacau ni brimo"
12460,f5.yg sudah bayar 50k 1.bpk kadim 200 2.humam 150 3.adi gendut 100 4.heri 50 5.pangat 50 6.ajis 50 7.ilham 50 8 8.faisal 50 9.ageng @ ono ceritane cah ndugal mlayu seko pacobaning urip.list baju 1.wono 19(l) 120 2 araujoo 27 (m) 120 3 ageng 12 m 4 piszz 23(s dewasa) 5 y. r 14 (m) 6 r. y 4 (l) 7 ilham 22 50 9.ageng 50 10.dewan 50 11.nanok 50 12.wawan 50 136556n6,f5.yg sudah bayar 50k 1.bpk kadim 200 2.humam 150 3.adi gendut 100 4.heri 50 5.pangat 50 6.ajis 50 7.ilham 50 8 8.faisal 50 9.ageng ono ceritane cah ndugal mlayu seko pacobaning urip.list baju 1.wono 19(l) 120 2 araujoo 27 (m) 120 3 ageng 12 m 4 piszz 23(s dewasa) 5 y. r 14 (m) 6 r. y 4 (l) 7 ilham 22 50 9.ageng 50 10.dewan 50 11.nanok 50 12.wawan 50 136556n6
12960,saya melakukan transfer di aplikasi brimo dengan menggunakan @nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata random dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii.... 9 jt,saya melakukan transfer di aplikasi brimo dengan menggunakan nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata random dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii.... 9 jt
15361,kenapa login gagal trs ya @brimo setelah download lagi di beda negara. boleh tolong saya kak,kenapa login gagal trs ya brimo setelah download lagi di beda negara. boleh tolong saya kak
16564,apk mobil banking paling parah! transfer sesama bri pake @ alias2 gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik,apk mobil banking paling parah! transfer sesama bri pake alias2 gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik
16629,"dapat notif : we detected something blocking the application screen. to make sure your data is safe, please close all popups and other overlays. if nothing is obstructing the application screen, try to restart the application and perform a malware scan. sudah sampai install ulang, reebo

remove punctuation

In [55]:
df["remove_punctuation"] = (
    df["remove_at"]
    .str.replace(r"[^\w\s]", " ", regex=True)
)

In [56]:
punctuation_changed = df[
    df["remove_at"] != df["remove_punctuation"]
][["remove_at", "remove_punctuation"]]

print(f"Rows affected: {len(punctuation_changed):,}")

punctuation_changed.head(10)

Rows affected: 37,945


,remove_at,remove_punctuation
0,"ribet banget ni apk sumpah dikir' verif dikit' verif udhmah nyedot pulsa mulu, tolong benerin lah, jangan ribet gini mau pake aja ribet cuh",ribet banget ni apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet gini mau pake aja ribet cuh
1,kenapa qris ga bisa di pakai ya?? daritadi loading mulu. transfer uang juga gabisa,kenapa qris ga bisa di pakai ya daritadi loading mulu transfer uang juga gabisa
2,"aplikasi nya sampah, kenapa tiba-tiba keluar terus registrasi ulang lagi, harus pake sms pulsa pula, sangat tidak membantu ,delete",aplikasi nya sampah kenapa tiba tiba keluar terus registrasi ulang lagi harus pake sms pulsa pula sangat tidak membantu delete
5,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue!,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue
7,"sinyal penuh, internet lancar tapi masih aj kode merah... aneh bener",sinyal penuh internet lancar tapi masih aj kode merah aneh bener
8,sinyal bagus tetap biru terus..membuat lama pembayaran,sinyal bagus tetap biru terus membuat lama pembayaran
11,"kenapa setelah masuk ke mbanking, harus aktivasi ulang ke cs buat transaksi finansialnya. padahal sebelumnya pernah aktif di hp yg sama. ribet banget tau gak sih!",kenapa setelah masuk ke mbanking harus aktivasi ulang ke cs buat transaksi finansialnya padahal sebelumnya pernah aktif di hp yg sama ribet banget tau gak sih
12,"kasik bintang 1 dulu aku new user bca, saldoku 96k pas di cek lagi kok berkurang jadi 71k ya? terus notif transaksi jga lambat, gaada catatan transaksi jga tolong dong di perbaiki. kalau udh dibalas nanti aku tambahin 1 bintang lagi",kasik bintang 1 dulu aku new user bca saldoku 96k pas di cek lagi kok berkurang jadi 71k ya terus notif transaksi jga lambat gaada catatan transaksi jga tolong dong di perbaiki kalau udh dibalas nanti aku tambahin 1 bintang lagi
14,tiap verifikasi wajah knapa slalu gagal. ganti hp karna hp lama udah eror. malah pas mau log in ke bca susah bgt. tiap verifikasi gagal terus. padahal pencahayaan udah aman.tolong bantuan dan sarqnnya dong,tiap verifikasi wajah knapa slalu gagal ganti hp karna hp lama udah eror malah pas mau log in ke bca susah bgt tiap verifikasi gagal terus padahal pencahayaan udah aman tolong bantuan dan sarqnnya dong
17,apa ini aplikasi bintang 1.. register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau... pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k


remove extra whitespace

In [57]:
df["cleaned"] = (
    df["remove_punctuation"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [58]:
# Normalisasi kata ulang dengan angka 2
# Contoh: tiba2 -> tiba-tiba, kadang2 -> kadang-kadang

def normalize_reduplicated_words(text):
    def replace_reduplication(match):
        word = match.group(1)
        return f"{word}-{word}"

    return re.sub(
        r"\b([a-zA-Z]+)2\b",
        replace_reduplication,
        str(text)
    )


df["before_reduplication"] = df["cleaned"]

df["cleaned"] = df["cleaned"].apply(
    normalize_reduplicated_words
)

df["reduplication_changed"] = (
    df["before_reduplication"] != df["cleaned"]
)

display(
    df.loc[
        df["reduplication_changed"],
        ["before_reduplication", "cleaned"]
    ].head(20)
)

,before_reduplication,cleaned
20,indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam2 pada hak sinyal bagus apk udah diperbarui kejadian terus berulang2 dan sering,indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam-berjam pada hak sinyal bagus apk udah diperbarui kejadian terus berulang-berulang dan sering
21,kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang2 trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya,kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang-ulang trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya
28,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar2 pake qris juga jadi males,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar-bayar pake qris juga jadi males
45,tolong update itu makin bener bukan makin lama merahnya klo akses apa aja transfer apa mutasi dikit2 sending muter2 doank,tolong update itu makin bener bukan makin lama merahnya klo akses apa aja transfer apa mutasi dikit-dikit sending muter-muter doank
57,mulai menyebalkan tiap buka bca mobil m bangking pasti mutar2 lama ga karuan padahal signal posel full ram aman saya coba ganti ke brimo langsung ga pake lama harap di perbaiki dong sistem nya bca,mulai menyebalkan tiap buka bca mobil m bangking pasti mutar-mutar lama ga karuan padahal signal posel full ram aman saya coba ganti ke brimo langsung ga pake lama harap di perbaiki dong sistem nya bca
104,saldo saya tiba2 ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan apk aneh,saldo saya tiba-tiba ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan apk aneh
117,kenapa lemot sekali mau buka apk nya pdhal jaringan bagus tpi jaringan terputus terus kira2 knpa ya apa lagi eror,kenapa lemot sekali mau buka apk nya pdhal jaringan bagus tpi jaringan terputus terus kira-kira knpa ya apa lagi eror
132,tau2 oneklik dinonaktifkan aja ngga jelas apa alasannya masa iya hanya sebulan ngga transaksi langsung diblokir ngeselin,tau-tau oneklik dinonaktifkan aja ngga jelas apa alasannya masa iya hanya sebulan ngga transaksi langsung diblokir ngeselin
140,kode sms tidak masuk2 saya sudah menunggu dan megirim kode sms secara berulang tp tidak kunjung masuk,kode sms tidak masuk-masuk saya sudah menunggu dan megirim kode sms secara berulang tp tidak kunjung masuk
141,verifikasi wajah sekarang susah banget ngga bisa2,verifikasi wajah sekarang susah banget ngga bisa-bisa


In [59]:
space_changed = df[
    df["remove_punctuation"] != df["cleaned"]
][["remove_punctuation", "cleaned"]]

print(f"Rows affected: {len(space_changed):,}")

space_changed.head(10)

Rows affected: 36,687


,remove_punctuation,cleaned
0,ribet banget ni apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet gini mau pake aja ribet cuh,ribet banget ni apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet gini mau pake aja ribet cuh
1,kenapa qris ga bisa di pakai ya daritadi loading mulu transfer uang juga gabisa,kenapa qris ga bisa di pakai ya daritadi loading mulu transfer uang juga gabisa
2,aplikasi nya sampah kenapa tiba tiba keluar terus registrasi ulang lagi harus pake sms pulsa pula sangat tidak membantu delete,aplikasi nya sampah kenapa tiba tiba keluar terus registrasi ulang lagi harus pake sms pulsa pula sangat tidak membantu delete
5,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue
7,sinyal penuh internet lancar tapi masih aj kode merah aneh bener,sinyal penuh internet lancar tapi masih aj kode merah aneh bener
8,sinyal bagus tetap biru terus membuat lama pembayaran,sinyal bagus tetap biru terus membuat lama pembayaran
11,kenapa setelah masuk ke mbanking harus aktivasi ulang ke cs buat transaksi finansialnya padahal sebelumnya pernah aktif di hp yg sama ribet banget tau gak sih,kenapa setelah masuk ke mbanking harus aktivasi ulang ke cs buat transaksi finansialnya padahal sebelumnya pernah aktif di hp yg sama ribet banget tau gak sih
12,kasik bintang 1 dulu aku new user bca saldoku 96k pas di cek lagi kok berkurang jadi 71k ya terus notif transaksi jga lambat gaada catatan transaksi jga tolong dong di perbaiki kalau udh dibalas nanti aku tambahin 1 bintang lagi,kasik bintang 1 dulu aku new user bca saldoku 96k pas di cek lagi kok berkurang jadi 71k ya terus notif transaksi jga lambat gaada catatan transaksi jga tolong dong di perbaiki kalau udh dibalas nanti aku tambahin 1 bintang lagi
14,tiap verifikasi wajah knapa slalu gagal ganti hp karna hp lama udah eror malah pas mau log in ke bca susah bgt tiap verifikasi gagal terus padahal pencahayaan udah aman tolong bantuan dan sarqnnya dong,tiap verifikasi wajah knapa slalu gagal ganti hp karna hp lama udah eror malah pas mau log in ke bca susah bgt tiap verifikasi gagal terus padahal pencahayaan udah aman tolong bantuan dan sarqnnya dong
17,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k


remove empty reviews

In [60]:
before = len(df)

df = df[
    df["cleaned"]
    .str.strip()
    .ne("")
].copy()

after = len(df)

print(f"Before : {before:,}")
print(f"After  : {after:,}")
print(f"Removed: {before-after:,}")

Before : 60,170
After  : 60,126
Removed: 44


comparison

In [61]:
comparison = df.copy()

comparison["changes"] = (
    (comparison["content"] != comparison["case_folding"]).astype(int) +
    (comparison["case_folding"] != comparison["unicode_normalized"]).astype(int) +
    (comparison["unicode_normalized"] != comparison["remove_emoji"]).astype(int) +
    (comparison["remove_emoji"] != comparison["remove_at"]).astype(int) +
    (comparison["remove_at"] != comparison["remove_punctuation"]).astype(int) +
    (comparison["remove_punctuation"] != comparison["cleaned"]).astype(int)
)

comparison = comparison.sort_values(
    by="changes",
    ascending=False
)

comparison[
    [
        "changes",
        "content",
        "case_folding",
        "unicode_normalized",
        "remove_emoji",
        "remove_at",
        "remove_punctuation",
        "cleaned"
    ]
].head(10)

,changes,content,case_folding,unicode_normalized,remove_emoji,remove_at,remove_punctuation,cleaned
29610,5,"Kami sekeluarga mempertanyakan 𝐅𝐮𝐧𝐠𝐬𝐢 dr app ini utk apa ❓ Ada saran utk menonaktifkan AKSESIBILITAS,lalu saya sdh mengikuti prosedur nya.Yang saya pertanyakan knpa tetap tdk bisa login ?! Letak kesalahan nya dimana ya","kami sekeluarga mempertanyakan 𝐅𝐮𝐧𝐠𝐬𝐢 dr app ini utk apa ❓ ada saran utk menonaktifkan aksesibilitas,lalu saya sdh mengikuti prosedur nya.yang saya pertanyakan knpa tetap tdk bisa login ?! letak kesalahan nya dimana ya","kami sekeluarga mempertanyakan Fungsi dr app ini utk apa ❓ ada saran utk menonaktifkan aksesibilitas,lalu saya sdh mengikuti prosedur nya.yang saya pertanyakan knpa tetap tdk bisa login ?! letak kesalahan nya dimana ya","kami sekeluarga mempertanyakan Fungsi dr app ini utk apa ada saran utk menonaktifkan aksesibilitas,lalu saya sdh mengikuti prosedur nya.yang saya pertanyakan knpa tetap tdk bisa login ?! letak kesalahan nya dimana ya","kami sekeluarga mempertanyakan Fungsi dr app ini utk apa ada saran utk menonaktifkan aksesibilitas,lalu saya sdh mengikuti prosedur nya.yang saya pertanyakan knpa tetap tdk bisa login ?! letak kesalahan nya dimana ya",kami sekeluarga mempertanyakan Fungsi dr app ini utk apa ada saran utk menonaktifkan aksesibilitas lalu saya sdh mengikuti prosedur nya yang saya pertanyakan knpa tetap tdk bisa login letak kesalahan nya dimana ya,kami sekeluarga mempertanyakan Fungsi dr app ini utk apa ada saran utk menonaktifkan aksesibilitas lalu saya sdh mengikuti prosedur nya yang saya pertanyakan knpa tetap tdk bisa login letak kesalahan nya dimana ya
40871,5,"Kenapa tiba-tiba terLog Out? kebetulan saya tidak punya kartu Debit ataupun Kredit dan kejadian ini hari Sabtu. Utk ke kantor cabang kan masih nunggu Senin. Posisi sy tidak terbiasa menyimpan uag cash, ya terpaksa tidak bisa beli apa² hanya karena livin yg terLog Out dg sendirinya😌","kenapa tiba-tiba terlog out? kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu. utk ke kantor cabang kan masih nunggu senin. posisi sy tidak terbiasa menyimpan uag cash, ya terpaksa tidak bisa beli apa² hanya karena livin yg terlog out dg sendirinya😌","kenapa tiba-tiba terlog out? kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu. utk ke kantor cabang kan masih nunggu senin. posisi sy tidak terbiasa menyimpan uag cash, ya terpaksa tidak bisa beli apa2 hanya karena livin yg terlog out dg sendirinya😌","kenapa tiba-tiba terlog out? kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu. utk ke kantor cabang kan masih nunggu senin. posisi sy tidak terbiasa menyimpan uag cash, ya terpaksa tidak bisa beli apa2 hanya karena livin yg terlog out dg sendirinya","kenapa tiba-tiba terlog out? kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu. utk ke kantor cabang kan masih nunggu senin. posisi sy tidak terbiasa menyimpan uag cash, ya terpaksa tidak bisa beli apa2 hanya karena livin yg terlog out dg sendirinya",kenapa tiba tiba terlog out kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu utk ke kantor cabang kan masih nunggu senin posisi sy tidak terbiasa menyimpan uag cash ya terpaksa tidak bisa beli apa2 hanya karena livin yg terlog out dg sendirinya,kenapa tiba tiba terlog out kebetulan saya tidak punya kartu debit ataupun kredit dan kejadian ini hari sabtu utk ke kantor cabang kan masih nunggu senin posisi sy tidak terbiasa menyimpan uag cash ya terpaksa tidak bisa beli apa-apa hanya karena livin yg terlog out dg sendirinya
21305,5,"kenapa dana sering hilang, tiba-tiba transaksi di tengah malam. jumlah juga lumayan bagi aku. ngelapor sana sini bukannya ada solusinya, bagus pake seabank aja, setiap ada masalah pasti pihak cs nya bertanggung jawab Sampai selesai‼️ cukup Allah yang tau, aku cari uang susah payah, malah seenaknya ngambil saldo","kenapa dana sering hilang, tiba-tiba transaksi di 

slang words normalization

In [62]:
slang_dict = pd.read_csv("../data/colloquial-indonesian-lexicon.csv")

slang_dict = slang_dict[["slang", "formal"]]

print(len(slang_dict))
slang_dict.head(20)

15006


,slang,formal
0,woww,wow
1,aminn,amin
2,met,selamat
3,netaas,menetas
4,keberpa,keberapa
5,eeeehhhh,eh
6,kata2nyaaa,kata-katanya
7,hallo,halo
8,kaka,kakak
9,ka,kak


In [63]:
slang_dict = dict(
    zip(
        slang_dict["slang"],
        slang_dict["formal"]
    )
)


In [64]:
print(slang_dict["blh"])

boleh


In [65]:
protected_words = {
    # Banks
    "bca",
    "bni",
    "bri",
    "mandiri",

    # Applications
    "brimo",
    "livin",
    "wondr",
    "blu",

    # Banking terms
    "qris",
    "qr",
    "atm",
    "otp",
    "pin",
    "rekening",
    "mbanking",
    "mbanking",
    "m-banking",
    "internet",
    "mobile",
    "token",
    "mtoken",

    # Company names
    "dana",
    "ovo",
    "gopay",
    "linkaja",
    "flip",

    # Payment systems
    "visa",
    "mastercard"
}

In [66]:
# from collections import Counter

# all_words = []

# for text in df["cleaned"]:
#     all_words.extend(text.split())

# word_freq = Counter(all_words)

# vocab_df = (
#     pd.DataFrame(
#         word_freq.items(),
#         columns=["word", "frequency"]
#     )
#     .sort_values(
#         by="frequency",
#         ascending=False
#     )
#         .reset_index(drop=True)
# )

In [67]:
# vocab_df["in_dictionary"] = vocab_df["word"].isin(slang_dict)

# vocab_df["formal"] = vocab_df["word"].map(slang_dict)

# vocab_df.head()

In [68]:
# def classify(row):
#     if row["in_dictionary"]:
#         return "Dictionary"

#     if row["frequency"] >= 20:
#         return "Review"

#     return "Rare"

# vocab_df["status"] = vocab_df.apply(classify, axis=1)

In [69]:
# vocab_df = vocab_df.sort_values(
#     by=[
#         "status",
#         "frequency"
#     ],
#     ascending=[True, False]
# )

In [70]:
# vocab_df["approved"] = ""
# vocab_df["notes"] = ""

# vocab_df.to_csv(
#     "vocabulary_review.csv",
#     index=False,
#     encoding="utf-8-sig"
# )


In [71]:
def normalize_text(text):
    words = text.split()

    normalized_words = []
    changes = 0

    for word in words:

        # Never normalize protected words
        if word in protected_words:
            normalized_words.append(word)
            continue

        new_word = slang_dict.get(word, word)

        if new_word != word:
            changes += 1

        normalized_words.append(new_word)

    return " ".join(normalized_words), changes

In [72]:
result = df["cleaned"].apply(normalize_text)

df["normalized"] = result.str[0]
df["normalization_changes"] = result.str[1]

print(f"Reviews normalized : {(df['normalization_changes']>0).sum():,}")

print()

print(df["normalization_changes"].describe())

Reviews normalized : 39,757

count    60126.000000
mean         2.046785
std          2.726933
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max         40.000000
Name: normalization_changes, dtype: float64


In [73]:
changed_reviews = df[
    df["normalization_changes"] > 0
][[
    "cleaned",
    "normalized",
    "normalization_changes"
]]

changed_reviews.head(20)

,cleaned,normalized,normalization_changes
0,ribet banget ni apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet gini mau pake aja ribet cuh,ribet banget nih apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet begini mau pakai saja ribet cuh,4
1,kenapa qris ga bisa di pakai ya daritadi loading mulu transfer uang juga gabisa,kenapa qris enggak bisa di pakai ya daritadi loading mulu transfer uang juga enggak bisa,2
2,aplikasi nya sampah kenapa tiba tiba keluar terus registrasi ulang lagi harus pake sms pulsa pula sangat tidak membantu delete,aplikasi nya sampah kenapa tiba tiba keluar terus registrasi ulang lagi harus pakai sms pulsa pula sangat tidak membantu delete,1
5,masa gue setiap isi flazz selalu gagal tapi kedebit gak layak bgt gue ini transaksi ke 4 gue,masa gue setiap isi flazz selalu gagal tapi kedebit enggak layak banget gue ini transaksi ke 4 gue,2
7,sinyal penuh internet lancar tapi masih aj kode merah aneh bener,sinyal penuh internet lancar tapi masih saja kode merah aneh benar,2
11,kenapa setelah masuk ke mbanking harus aktivasi ulang ke cs buat transaksi finansialnya padahal sebelumnya pernah aktif di hp yg sama ribet banget tau gak sih,kenapa setelah masuk ke mbanking harus aktivasi ulang ke cs buat transaksi finansialnya padahal sebelumnya pernah aktif di hp yang sama ribet banget tau enggak sih,2
12,kasik bintang 1 dulu aku new user bca saldoku 96k pas di cek lagi kok berkurang jadi 71k ya terus notif transaksi jga lambat gaada catatan transaksi jga tolong dong di perbaiki kalau udh dibalas nanti aku tambahin 1 bintang lagi,kasik bintang 1 dulu aku new user bca saldoku 96k pas di cek lagi kok berkurang jadi 71k ya terus notif transaksi juga lambat enggak ada catatan transaksi juga tolong dong di perbaiki kalau sudah dibalas nanti aku tambahkan 1 bintang lagi,5
13,ini bca lgi gangguan apa gimana saya loading terus mau masuk ke bca mobile,ini bca lagi gangguan apa bagaimana saya loading terus mau masuk ke bca mobile,2
14,tiap verifikasi wajah knapa slalu gagal ganti hp karna hp lama udah eror malah pas mau log in ke bca susah bgt tiap verifikasi gagal terus padahal pencahayaan udah aman tolong bantuan dan sarqnnya dong,tiap verifikasi wajah knapa selalu gagal ganti hp karena hp lama sudah eror malah pas mau log ini ke bca susah banget tiap verifikasi gagal terus padahal pencahayaan sudah aman tolong bantuan dan sarqnnya dong,6
15,setelah d update malah minta masukin norek seperti awal install bca trus ga bs d ceklist buat lanjutin k proses selanjutnya ini gmn sih sistemnya bukannya memudahkan tp malah menyulitkan padahal sy sll menggunakan bca dlm setiap transaksi,setelah di update malah meminta memasuki norek seperti awal install bca terus enggak bisa di ceklis buat lanjutkan ke proses selanjutnya ini bagaimana sih sistemnya bukannya memudahkan tapi malah menyulitkan padahal saya selalu menggunakan bca dalam setiap transaksi,15


Reduce three or more consecutive identical characters  
example = "paraaahhhhhh" , "jeeelleeekkkk"

In [74]:
def normalize_repeated_chars(text):
    # Reduce 3 or more repeated characters to a single character
    return re.sub(r"(.)\1{2,}", r"\1", text)

def remove_repeated_words(text):
    words = text.split()
    cleaned_words = []

    for word in words:
        if not cleaned_words or word != cleaned_words[-1]:
            cleaned_words.append(word)

    return " ".join(cleaned_words)

def remove_single_character_noise(text):
    return " ".join(
        word for word in text.split()
        if len(word) > 1
    )

df["final_text"] = (
    df["normalized"]
    .apply(normalize_repeated_chars)
    .apply(remove_repeated_words)
    .apply(remove_single_character_noise)
)

In [75]:
df["final_text"] = (
    df["normalized"]
    .apply(normalize_repeated_chars)
    .apply(remove_repeated_words)
    .apply(remove_single_character_noise)
)

In [76]:
df["before_repeat"] = df["normalized"]

def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1", text)

df["after_repeat"] = df["normalized"].apply(normalize_repeated_chars)

show = df[
    df["before_repeat"] != df["after_repeat"]
][["before_repeat", "after_repeat"]]

print(f"Jumlah review yang berubah: {len(show):,}")
show.head(20)

Jumlah review yang berubah: 3,068


,before_repeat,after_repeat
19,makin paraaaaaah setelah diupdate aplikasi nya ya allah,makin parah setelah diupdate aplikasi nya ya allah
64,apk baru diupdate malah enggak bisa dibuka kan aneh apk ngennn,apk baru diupdate malah enggak bisa dibuka kan aneh apk ngen
101,kinerja bank nya bagus aplilkasi nya lemot kalau buka qris diperbaiki yang komplain sudah banyak kok masih bilang enggak ada kendala haduhhhh mau jadi bank bumn enggak mau terima kritik 24 12 sudah berapa minggu enggak ada perbaikan qris mu itu mu pecat saja,kinerja bank nya bagus aplilkasi nya lemot kalau buka qris diperbaiki yang komplain sudah banyak kok masih bilang enggak ada kendala haduh mau jadi bank bumn enggak mau terima kritik 24 12 sudah berapa minggu enggak ada perbaikan qris mu itu mu pecat saja
102,jangan terlalu percaya dengan aplikasi saat urgen lampu indikatornya merah terussss kan goblog,jangan terlalu percaya dengan aplikasi saat urgen lampu indikatornya merah terus kan goblog
161,ini bagaimana sih bca tolong dong yang jelas masa potongan bulanan potongnnya enggak jls benar bulan kemarin saldo saya di sedot 25000 sekarang malah 24000 tapi di prosedurnya potongan setiap bulan hanya 15000 bagaimana ini bca aneh,ini bagaimana sih bca tolong dong yang jelas masa potongan bulanan potongnnya enggak jls benar bulan kemarin saldo saya di sedot 250 sekarang malah 240 tapi di prosedurnya potongan setiap bulan hanya 150 bagaimana ini bca aneh
162,ih paling kesel sama bca indikator merah tusss lama lagi ayok di perbaiki padahal sinyal bagus kebiasaan setiap mau pembayaran merah teusss lamaaa lagi,ih paling kesel sama bca indikator merah tus lama lagi ayok di perbaiki padahal sinyal bagus kebiasaan setiap mau pembayaran merah teus lama lagi
168,bca sebagai bank no 1 di indonesia ternyata tidak bisa memberi penjelasan kemana hilangnya uang saya yang tidak tercantum dalam mutasi memang nominalnya hanya 30 000 tapi itu jelas membuktikan keteledoran pihak bca dalam menjaga uang nasabahnya bagaimana jika 30 000 dikalikan ribuan orang lain yang enggak sadar uangnya terpotong enggak jelas tolong bca jelaskan kemana uang saya itu,bca sebagai bank no 1 di indonesia ternyata tidak bisa memberi penjelasan kemana hilangnya uang saya yang tidak tercantum dalam mutasi memang nominalnya hanya 30 0 tapi itu jelas membuktikan keteledoran pihak bca dalam menjaga uang nasabahnya bagaimana jika 30 0 dikalikan ribuan orang lain yang enggak sadar uangnya terpotong enggak jelas tolong bca jelaskan kemana uang saya itu
186,ini kenapa ya setiap mau masuk apk selalu ada tulisan 205 transaksi tidak dapat diproses coba lagi nanti kenapaaaaa saya mau cek saldo mau transfer padahal sinyal bagus sudah beberapa kali restart hp tolong diperbaiki secepatnya,ini kenapa ya setiap mau masuk apk selalu ada tulisan 205 transaksi tidak dapat diproses coba lagi nanti kenapa saya mau cek saldo mau transfer padahal sinyal bagus sudah beberapa kali restart hp tolong diperbaiki secepatnya
217,wahaii pemuda bikin aplikasi publik yang benerlah tiap buka aplikasi meminta verifikasi sms mulu lagi bikin kuis jari jariiiii lu meminta sms terus,wahaii pemuda bikin aplikasi publik yang benerlah tiap buka aplikasi meminta verifikasi sms mulu lagi bikin kuis jari jari lu meminta sms terus
227,seusai di update jadi leeeemoootttt,seusai di update jadi lemot


In [77]:
final_dataset = df[
    [
        "reviewId",
        "bank",
        "score",
        "year",
        "final_text"
    ]
].copy()

final_dataset.rename(
    columns={
        "final_text": "text"
    },
    inplace=True
)

final_dataset.head(30)

,reviewId,bank,score,year,text
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,BCAMOBILE_REVIEWS,1,2025,ribet banget nih apk sumpah dikir verif dikit verif udhmah nyedot pulsa mulu tolong benerin lah jangan ribet begini mau pakai saja ribet cuh
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,BCAMOBILE_REVIEWS,2,2025,kenapa qris enggak bisa di pakai ya daritadi loading mulu transfer uang juga enggak bisa
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,BCAMOBILE_REVIEWS,1,2025,aplikasi nya sampah kenapa tiba keluar terus registrasi ulang lagi harus pakai sms pulsa pula sangat tidak membantu delete
3,32d28ac6-c538-4749-969f-8af060915d96,BCAMOBILE_REVIEWS,3,2025,bagus
4,f99259b7-0d45-4422-b667-6c4fd521638b,BCAMOBILE_REVIEWS,1,2025,tidak ada solusi ketika ada kendala di persulit di lempar kesana kesini sangat mengecewakan harusnya respon dan solusi yang di berikan
5,1b50a316-798a-42a3-8bfe-0154853d0f4a,BCAMOBILE_REVIEWS,1,2025,masa gue setiap isi flazz selalu gagal tapi kedebit enggak layak banget gue ini transaksi ke gue
6,6ac413fd-0745-430c-b545-da39c54ff60e,BCAMOBILE_REVIEWS,1,2025,mau aktifasi di handphone baru semakin ribet dan sulit tidak seperti yang lalu
7,8fce3e87-d488-4ed5-9f91-17cdae1a6b5f,BCAMOBILE_REVIEWS,1,2025,sinyal penuh internet lancar tapi masih saja kode merah aneh benar
8,b8568501-8111-48cc-9fd9-eefda3fdd1d6,BCAMOBILE_REVIEWS,2,2025,sinyal bagus tetap biru terus membuat lama pembayaran
9,06df10c9-69d4-4589-bd3d-505443d4fd53,BCAMOBILE_REVIEWS,1,2025,saya mau login menggunakan pulsa kode selalu tidak keluar


In [78]:
final_dataset.shape

(60126, 5)

In [79]:
final_dataset.to_csv("preprocessed_data_2025.csv", index=False)